# Mosteller and Wallace: The Federalist Papers

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* The founding authorship study; Bayesian reasoning on function words

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09f_mosteller_and_wallace.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Modules 9a–9e taught the machinery of stylometry. This notebook is where it came
from.

In 1963 Frederick Mosteller and David Wallace attacked a question historians had
argued over for 175 years: who wrote the twelve *Federalist Papers* claimed by
both Alexander Hamilton and James Madison? Their answer — Madison, with odds they
put at roughly 100:1 for the least clear case, No. 55 — was built on the rates of
ordinary function words like *by*, *from* and *to*.

It established three things this course has taken for granted: that function
words carry authorship, that the reasoning should be probabilistic, and that
attribution can be settled quantitatively at all.

## The tooling

Mosteller and Wallace worked with punched cards and hand-computed Bayes factors.
The modern equivalents are R's **stylo** (the field standard) and Python's
**faststylometry**; `scikit-learn`'s `MultinomialNB` implements the same
Bayesian core in three lines. We build it directly, because the reasoning is the
lesson.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# We cannot ship the Federalist, so we reproduce the METHOD on the corpus we
# have: two authors with several works each, and a disputed text held out.
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
         .apply(" ".join).reset_index())
works["n"] = works["text"].map(lambda t: len(tokenise(t)))
works = works[works["n"] >= 5000]
counts = works["author"].value_counts()
pair = counts[counts >= 3].index[:2].tolist()
print(f"standing in for Hamilton and Madison: {pair[0]} and {pair[1]}")
sel = works[works["author"].isin(pair)].reset_index(drop=True)
print(sel.groupby("author").size().to_string())

## The function words they chose

Mosteller and Wallace's insight was to use words with **no topical content** —
words whose rate cannot be explained by what the text is about, only by who
wrote it. Content words would have told them which paper discussed taxation.

In [ ]:
FUNCTION_WORDS = ["a", "all", "also", "an", "and", "any", "are", "as", "at",
                  "be", "been", "but", "by", "can", "do", "down", "even", "every",
                  "for", "from", "had", "has", "have", "her", "his", "if", "in",
                  "into", "is", "it", "its", "may", "more", "must", "my", "no",
                  "not", "now", "of", "on", "one", "only", "or", "our", "shall",
                  "should", "so", "some", "such", "than", "that", "the", "their",
                  "then", "there", "things", "this", "to", "up", "upon", "was",
                  "were", "what", "when", "which", "who", "will", "with", "would"]
print(f"{len(FUNCTION_WORDS)} function words, in the spirit of the 1963 list")

def rates(text):
    toks = tokenise(text)
    c = Counter(toks)
    n = max(len(toks), 1)
    return np.array([c[w] / n * 1000 for w in FUNCTION_WORDS])

R = np.vstack([rates(t) for t in sel["text"]])
print(f"feature matrix: {R.shape[0]} works x {R.shape[1]} rates per 1,000 words")

In [ ]:
# Which words separate the two authors most? This is their Table 2.
a_mask = (sel["author"] == pair[0]).values
diff = R[a_mask].mean(0) - R[~a_mask].mean(0)
order = np.argsort(np.abs(diff))[::-1][:12]
print(f"{'word':<12}{pair[0][:16]:>18}{pair[1][:16]:>18}{'difference':>13}")
print("-" * 62)
for i in order:
    print(f"{FUNCTION_WORDS[i]:<12}{R[a_mask][:, i].mean():>18.2f}"
          f"{R[~a_mask][:, i].mean():>18.2f}{diff[i]:>13.2f}")
print()
print("Mosteller and Wallace found 'upon' did most of the work: Hamilton used it")
print("constantly, Madison almost never. Nobody chooses how often to write")
print("'upon' — that unconsciousness is exactly why it is evidence.")

In [ ]:
# Their Bayesian argument, in miniature: leave one work out, and ask which
# author's profile makes the observed rates more probable.
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import LeaveOneOut

y = sel["author"].values
loo = LeaveOneOut()
correct, odds = 0, []
for tr, te in loo.split(R):
    clf = GaussianNB().fit(R[tr], y[tr])
    prob = clf.predict_proba(R[te])[0]
    pred = clf.classes_[prob.argmax()]
    correct += int(pred == y[te][0])
    odds.append(prob.max() / max(prob.min(), 1e-12))
print(f"leave-one-out accuracy: {correct}/{len(R)} = {correct/len(R):.3f}")
print(f"median odds for the winning author: {np.median(odds):,.0f} : 1")
print()
print("Odds, not p-values. That framing was unusual in 1963 and is why the")
print("study reads as modern: it reports how much more probable one authorship")
print("is than the other, which is the question a historian actually asked.")

plt.figure(figsize=(6.5, 3.6))
plt.hist(np.log10(np.clip(odds, 1, 1e12)), bins=12, color="steelblue")
plt.xlabel("log10 odds for the attributed author")
plt.ylabel("works"); plt.title("Strength of evidence, work by work")
plt.tight_layout()

## The case is not closed

A 2025 Bayesian-clustering reanalysis (arXiv:2511.16040) found disputed essays
55 and 62 grouping with 43 known Hamilton essays — "counter to the analysis of
Mosteller and Wallace (1963)" — suggesting more Hamilton involvement, or
co-authorship, than the classic study allowed.

Sixty years on, with more method and more compute, the canonical result is under
revision. That is worth holding onto: an attribution is a probabilistic claim
about evidence, not a fact established once. Report your odds, your feature set
and your assumptions, so the next person can do to you what that paper did to
Mosteller and Wallace.

### 🔧 Your turn 1

Drop 'upon' and the five strongest discriminating words from FUNCTION_WORDS and re-run. How much does accuracy fall?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually less than people expect, because function-word evidence is distributed
rather than concentrated — dozens of small consistent differences, not one
magic word. That redundancy is what makes the method robust to an author
consciously avoiding a tell, and it is the reason Burrows's Delta uses hundreds
of words rather than a handful. If your own attribution result collapses when
you remove one feature, it was resting on that feature, and you should say so.

</details>